In [1]:
import pandas as pd

movies = pd.read_csv('../data/movielens_engineered.csv')
movies.head()

,movieId,title,genres,avg_rating,rating_count,all_tags,imdbId,tmdbId,year,num_genres,...,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western,tag_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215.0,fun pixar,114709,862.0,1995.0,5,...,0,0,0,0,0,0,0,0,0,2
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.431818,110.0,Robin Williams fantasy game magic board game,113497,8844.0,1995.0,3,...,0,0,0,0,0,0,0,0,0,7
2,3,Grumpier Old Men (1995),Comedy|Romance,3.259615,52.0,moldy old,113228,15602.0,1995.0,2,...,0,0,0,0,1,0,0,0,0,2
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,7.0,NaN,114885,31357.0,1995.0,3,...,0,0,0,0,1,0,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,3.071429,49.0,pregnancy remake,113041,11862.0,1995.0,1,...,0,0,0,0,0,0,0,0,0,2


### cleaning stuff up

movieId is just a random number assigned to each movie so theres no point trying to predict that, it doesnt actually mean anything. also title/genres(the text version)/imdbId/tmdbId are just ids/text so we wont use those as features.

instead lets make our own target: was the movie rated highly or not (>=3.5 avg rating). this way we actually have something worth predicting

In [2]:
# quick look at what's missing before we do anything
movies.info()
movies.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movieId            9742 non-null   int64  
 1   title              9742 non-null   str    
 2   genres             9742 non-null   str    
 3   avg_rating         9724 non-null   float64
 4   rating_count       9724 non-null   float64
 5   all_tags           1572 non-null   str    
 6   imdbId             9742 non-null   int64  
 7   tmdbId             9734 non-null   float64
 8   year               9729 non-null   float64
 9   num_genres         9742 non-null   int64  
 10  genre_Action       9742 non-null   int64  
 11  genre_Adventure    9742 non-null   int64  
 12  genre_Animation    9742 non-null   int64  
 13  genre_Children     9742 non-null   int64  
 14  genre_Comedy       9742 non-null   int64  
 15  genre_Crime        9742 non-null   int64  
 16  genre_Documentary  9742 non-null   

movieId                 0
title                   0
genres                  0
avg_rating             18
rating_count           18
all_tags             8170
imdbId                  0
tmdbId                  8
year                   13
num_genres              0
genre_Action            0
genre_Adventure         0
genre_Animation         0
genre_Children          0
genre_Comedy            0
genre_Crime             0
genre_Documentary       0
genre_Drama             0
genre_Fantasy           0
genre_Film-Noir         0
genre_Horror            0
genre_IMAX              0
genre_Musical           0
genre_Mystery           0
genre_Romance           0
genre_Sci-Fi            0
genre_Thriller          0
genre_War               0
genre_Western           0
tag_count               0
dtype: int64

In [3]:
# cant predict a rating bucket for movies that dont have a rating lol, so drop those
movies = movies.dropna(subset=['avg_rating', 'rating_count']).copy()

# year and tag_count missing just means we dont know it / there were no tags, fill with something sensible
movies['year'] = movies['year'].fillna(movies['year'].median())
movies['tag_count'] = movies['tag_count'].fillna(0)

movies.isna().sum()

movieId                 0
title                   0
genres                  0
avg_rating              0
rating_count            0
all_tags             8170
imdbId                  0
tmdbId                  8
year                    0
num_genres              0
genre_Action            0
genre_Adventure         0
genre_Animation         0
genre_Children          0
genre_Comedy            0
genre_Crime             0
genre_Documentary       0
genre_Drama             0
genre_Fantasy           0
genre_Film-Noir         0
genre_Horror            0
genre_IMAX              0
genre_Musical           0
genre_Mystery           0
genre_Romance           0
genre_Sci-Fi            0
genre_Thriller          0
genre_War               0
genre_Western           0
tag_count               0
dtype: int64

In [4]:
# making the target column - 1 if its a good movie (rating wise), 0 if not
movies['good_movie'] = (movies['avg_rating'] >= 3.5).astype(int)
movies['good_movie'].value_counts()

good_movie
0    5044
1    4680
Name: count, dtype: int64

In [5]:
# grabbing all the genre columns (they're already one hot encoded for us) plus a few other useful columns
genre_columns = [col for col in movies.columns if col.startswith('genre_')]
cols_to_use = genre_columns + ['year', 'num_genres', 'tag_count', 'rating_count']

X = movies[cols_to_use]
y = movies['good_movie']
X.head()

,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,...,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western,year,num_genres,tag_count,rating_count
0,0,1,1,1,1,0,0,0,1,0,...,0,0,0,0,0,0,1995.0,5,2,215.0
1,0,1,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0,1995.0,3,7,110.0
2,0,0,0,0,1,0,0,0,0,0,...,0,1,0,0,0,0,1995.0,2,2,52.0
3,0,0,0,0,1,0,0,1,0,0,...,0,1,0,0,0,0,1995.0,3,0,7.0
4,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1995.0,1,2,49.0


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# SVMs care a lot about scale so gotta standardize everything first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape

((7779, 23), (1945, 23))

### alright, time to actually train the svm

In [7]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

svm_clf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_clf.fit(X_train_scaled, y_train)

preds = svm_clf.predict(X_test_scaled)

print('accuracy:', accuracy_score(y_test, preds))
print()
print(classification_report(y_test, preds))
print(confusion_matrix(y_test, preds))

accuracy: 0.6663239074550128

              precision    recall  f1-score   support

           0       0.67      0.69      0.68      1009
           1       0.66      0.64      0.65       936

    accuracy                           0.67      1945
   macro avg       0.67      0.67      0.67      1945
weighted avg       0.67      0.67      0.67      1945

[[701 308]
 [341 595]]


### which kernel actually works best here?

rbf isn't always the winner so let's just try a few and see what sticks

In [8]:
for k in ['linear', 'rbf', 'poly']:
    clf = SVC(kernel=k, random_state=42)
    clf.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test_scaled))
    print(k, '->', round(acc, 4))

linear -> 0.6257


rbf -> 0.6663


poly -> 0.656
